# Step 4: Recurrent Neural Networks (RNN)

An MLP and CNN both process inputs of **fixed size**. Language, speech, and time series
are sequences of variable length — and the meaning of each element depends on what came before.

An RNN processes sequences by maintaining a **hidden state** that is updated at each step:

```
hₜ = tanh(Wₓₕ xₜ + Wₕₕ hₜ₋₁ + b)
yₜ = Wₕᵧ hₜ
```

The same weights (Wₓₕ, Wₕₕ, Wₕᵧ) are reused at every time step — like weight sharing
in space for CNNs, but over time.

**Task**: Character-level language model — predict the next character given a sequence of previous characters.

**What you'll see:**
1. RNN forward pass through a sequence
2. Training on a small text corpus
3. Generating text by sampling from the model
4. Exploding and vanishing gradients in practice — measured directly

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
plt.rcParams['figure.dpi'] = 100

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Data: A Small Text Corpus

We train on a small excerpt. The model learns the probability of each character
given the preceding context. At inference, we sample character by character to generate text.

In [ ]:
# Small training corpus — enough to see structure emerge
text = """To be or not to be that is the question
Whether tis nobler in the mind to suffer
The slings and arrows of outrageous fortune
Or to take arms against a sea of troubles
And by opposing end them to die to sleep
No more and by a sleep to say we end
The heartache and the thousand natural shocks
That flesh is heir to tis a consummation
Devoutly to be wished to die to sleep
To sleep perchance to dream aye theres the rub
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil
Must give us pause there is the respect
That makes calamity of so long life"""

# Build character-level vocabulary
chars = sorted(set(text))
vocab_size = len(chars)
char2idx = {c: i for i, c in enumerate(chars)}
idx2char = {i: c for c, i in char2idx.items()}

# Encode text as integer indices
data = torch.tensor([char2idx[c] for c in text], dtype=torch.long)

print(f"Text length: {len(text)} characters")
print(f"Vocabulary size: {vocab_size} unique characters")
print(f"Characters: {repr(''.join(chars))}")

def make_batches(data, seq_len, batch_size):
    """Create (input, target) pairs: target is input shifted one position right."""
    n = (len(data) - 1) // (seq_len * batch_size)
    data = data[:n * seq_len * batch_size + 1]
    x = data[:-1].view(batch_size, -1)  # (batch_size, n*seq_len)
    y = data[1:].view(batch_size, -1)
    return x, y

## The RNN Architecture

```
For each time step t:
  hₜ = tanh(Wₓₕ · xₜ + Wₕₕ · hₜ₋₁ + b_h)   # hidden state update
  ŷₜ = Wₕᵧ · hₜ + b_y                          # output logits

Parameters:
  Wₓₕ: (vocab_size → hidden)  — encodes the current input
  Wₕₕ: (hidden → hidden)      — passes information forward in time
  Wₕᵧ: (hidden → vocab_size)  — decodes hidden state to output distribution
```

The **same weights are used at every time step** — this is weight sharing over time.
Gradient flows backward through time by unrolling the recurrence.

In [ ]:
class VanillaRNN(nn.Module):
    def __init__(self, vocab_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # Core RNN cell: input + previous hidden → next hidden
        self.rnn = nn.RNN(vocab_size, hidden_size, batch_first=True, nonlinearity='tanh')
        # Output projection: hidden → logits over vocabulary
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x, h=None):
        # x: (batch, seq_len) of token indices
        # One-hot encode inputs
        x_onehot = torch.zeros(x.size(0), x.size(1), vocab_size, device=x.device)
        x_onehot.scatter_(2, x.unsqueeze(2), 1.0)

        out, h_new = self.rnn(x_onehot, h)  # out: (batch, seq_len, hidden)
        logits = self.fc(out)               # (batch, seq_len, vocab_size)
        return logits, h_new

    def generate(self, seed_char, length=200, temperature=1.0):
        """Sample characters autoregressively."""
        self.eval()
        generated = [seed_char]
        h = None
        x = torch.tensor([[char2idx[seed_char]]], device=device)
        with torch.no_grad():
            for _ in range(length):
                logits, h = self.forward(x, h)
                probs = torch.softmax(logits[0, -1] / temperature, dim=0)
                next_idx = torch.multinomial(probs, 1).item()
                generated.append(idx2char[next_idx])
                x = torch.tensor([[next_idx]], device=device)
        return ''.join(generated)

hidden_size = 128
model = VanillaRNN(vocab_size, hidden_size).to(device)
params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {params:,}")
print(f"  Wxh: {vocab_size}×{hidden_size} = {vocab_size*hidden_size:,}")
print(f"  Whh: {hidden_size}×{hidden_size} = {hidden_size*hidden_size:,}")
print(f"  Why: {hidden_size}×{vocab_size} = {hidden_size*vocab_size:,}")

In [ ]:
# Training
seq_len = 40
batch_size = 4
x_data, y_data = make_batches(data, seq_len, batch_size)

optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
criterion = nn.CrossEntropyLoss()

losses = []
grad_norms = []   # Track gradient norms to observe vanishing/exploding gradients
n_epochs = 300

for epoch in range(n_epochs):
    model.train()
    # Pick a random chunk of the sequence each epoch
    start = np.random.randint(0, x_data.size(1) - seq_len)
    x_batch = x_data[:, start:start+seq_len].to(device)
    y_batch = y_data[:, start:start+seq_len].to(device)

    logits, _ = model(x_batch)
    # logits: (batch, seq_len, vocab_size) → reshape for cross-entropy
    loss = criterion(logits.reshape(-1, vocab_size), y_batch.reshape(-1))

    optimizer.zero_grad()
    loss.backward()

    # Measure gradient norm at Whh (the recurrent weight) before clipping
    whh_grad = model.rnn.weight_hh_l0.grad
    if whh_grad is not None:
        grad_norms.append(whh_grad.norm().item())

    # Gradient clipping — standard practice for RNNs; prevents exploding gradients
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1:3d} | loss: {loss.item():.4f} | "
              f"Wₕₕ grad norm: {grad_norms[-1]:.4f}")

In [ ]:
# Plot training loss and gradient norms
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Smooth loss
window = 20
smooth_loss = np.convolve(losses, np.ones(window)/window, mode='valid')
ax1.plot(losses, alpha=0.3, color='steelblue', lw=1, label='raw')
ax1.plot(smooth_loss, color='steelblue', lw=2, label=f'{window}-step avg')
ax1.set_xlabel('Training step'); ax1.set_ylabel('Cross-entropy loss')
ax1.set_title('Training loss (vanilla RNN)')
ax1.legend(); ax1.grid(True, alpha=0.3)
ax1.axhline(np.log(vocab_size), color='tomato', linestyle=':', label='random chance')
ax1.legend()

# Gradient norms
ax2.semilogy(grad_norms, color='tomato', lw=1.5, alpha=0.7)
ax2.set_xlabel('Training step'); ax2.set_ylabel('||Wₕₕ gradient|| (log scale)')
ax2.set_title('Wₕₕ gradient norm — spikes = exploding, trend = vanishing')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Generate text at different temperatures
print("=== Generated Text ===")
for temp in [0.5, 1.0, 1.5]:
    generated = model.generate('T', length=150, temperature=temp)
    print(f"\nTemperature={temp}:")
    print(generated)
    print()

## The Vanishing Gradient Problem in RNNs

Backpropagation **through time** (BPTT) unrolls the recurrence and applies the chain rule.
The gradient of the loss at step T with respect to the hidden state at step t is:

```
∂L_T/∂hₜ = ∂L_T/∂h_T · Π_{k=t+1}^{T} (∂h_k/∂h_{k-1})
           = ∂L_T/∂h_T · Π_{k=t+1}^{T} (Wₕₕᵀ · diag(tanh'(z_k)))
```

This is a product of (T-t) matrices. If the largest eigenvalue of Wₕₕ is < 1,
the product decays exponentially. If > 1, it explodes.

In practice: **gradients vanish for dependencies beyond ~10-20 steps**.
The RNN forgets its own history.

In [ ]:
# Visualize gradient magnitude vs. distance from the loss
# Create a long sequence and measure how much gradient reaches each position

model.eval()
seq_len_test = 60
x_test = data[:seq_len_test].unsqueeze(0).to(device)  # (1, 60)
y_test = data[1:seq_len_test+1].unsqueeze(0).to(device)

# Enable gradient computation
x_onehot = torch.zeros(1, seq_len_test, vocab_size, device=device, requires_grad=True)
x_onehot.data.scatter_(2, x_test.unsqueeze(2), 1.0)

out, _ = model.rnn(x_onehot)
logits = model.fc(out)
loss = criterion(logits.reshape(-1, vocab_size), y_test.reshape(-1))
loss.backward()

# Gradient magnitude at each time step's input
input_grads = x_onehot.grad.norm(dim=2).squeeze(0).detach().cpu().numpy()

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(seq_len_test), input_grads, color='steelblue', alpha=0.8)
ax.set_xlabel('Time step (position in sequence)')
ax.set_ylabel('Gradient magnitude at input')
ax.set_title('How much gradient reaches each position — the further from the loss, the less')
ax.grid(True, alpha=0.3, axis='y')

# Annotate the vanishing
ax.annotate('Loss computed\nat final step',
    xy=(seq_len_test-1, input_grads[-1]), xytext=(45, input_grads.max()*0.7),
    arrowprops=dict(arrowstyle='->', color='tomato'), color='tomato', fontsize=10)

plt.tight_layout()
plt.show()
print(f"Gradient at step 0 vs step {seq_len_test-1}: ratio = {input_grads[0]/input_grads[-1]:.4f}")

## Key Takeaways

| Concept | What we learned |
|---|---|
| **Hidden state** | Carries information forward through time; initialized to zero |
| **Weight sharing over time** | Same Wₓₕ, Wₕₕ, Wₕᵧ at every step — like CNN over space |
| **BPTT** | Unroll recurrence, apply chain rule back through time |
| **Vanishing gradient** | Product of Wₕₕ matrices decays if max eigenvalue < 1 |
| **Exploding gradient** | Same product explodes if eigenvalue > 1; fixed by gradient clipping |
| **Temperature** | Controls sampling randomness: low = conservative, high = creative |

**The core problem**: vanilla RNNs can't reliably learn dependencies longer than ~20 steps.
They forget their early inputs. The fix: **LSTM** — gated cell state that preserves
information across arbitrarily long sequences.